# 09 - Generalization Checks

**Project:** Causal ML for Climate Policy Effectiveness
**Purpose:** Two extensions requested after reviewing the main finding:

**Part A - Mechanism investigation.** Does the pre-trend violation found in
Notebook 06 look like income convergence (treated countries already on a
different income trajectory, which could mechanically produce a different
emissions trajectory) rather than anticipation of the policy itself?

**Part B - Generalization check.** Does the same naive-looks-robust /
fails-under-sensitivity pattern show up for a genuinely different treatment
definition, or is it specific to the combined "either tax or ETS" treatment
used in the main analysis? Carbon tax adoption (26 countries, staggered
across 17 distinct years, no single dominant cohort) is used as the second
treatment -- reusing the same already-vetted WCPD data rather than sourcing
a new treatment dataset, which would reintroduce exactly the construct-
validity risk this project's rebuild was designed to avoid.

**Input:** `data/processed/analysis_panel.csv`, `data/processed/treatment_matrix.csv`
**Output:** `outputs/tables/mechanism_income_check.csv`, `outputs/tables/tax_only_*.csv`

In [1]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import honestdid

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.did_cs import run_callaway_santanna
from src.event_study import run_event_study, pretrend_test
from src.visualization import set_publication_style, save_figure, COLORS

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
set_publication_style()

panel = pd.read_csv(PROCESSED_DIR / "analysis_panel.csv")
COVARIATES = ["population", "gdp_per_capita", "urban_pct"]

---
## Part A: Income-Convergence Mechanism Check

If treated countries were already converging toward higher income (and higher
income is associated with a different emissions trajectory, e.g. structural
shift toward services), that could produce a pre-trend in emissions that has
nothing to do with anticipating carbon pricing. Two checks: (A1) compare
pre-period GDP per capita growth between treated and control countries;
(A2) re-run the event study with log(GDP per capita) itself as the outcome,
to see if income shows the same rising-then-flattening shape around adoption
that emissions did.

In [2]:
print("=" * 60)
print("A1 - Pre-period GDP per capita growth: treated vs. control")
print("=" * 60)

panel_a = panel.dropna(subset=["log_co2", "gdp_per_capita"] + COVARIATES).copy()
adoption_year = panel_a.loc[panel_a["treat_carbon_price"] == 1].groupby("iso3")["year"].min()
panel_a["adoption_year"] = panel_a["iso3"].map(adoption_year)

# Controls have no adoption event, so their comparison window must be a
# fixed, consistently-chosen set of calendar years (NOT a random sample of
# rows, which breaks chronological order and makes the resulting "growth"
# figure meaningless) -- the 5 years before the median treated adoption year.
median_adoption_year = int(adoption_year.median())
rows = []
for iso3, grp in panel_a.groupby("iso3"):
    grp = grp.sort_values("year")
    ay = adoption_year.get(iso3, np.nan)
    if pd.notna(ay):
        pre = grp[(grp["year"] >= ay - 5) & (grp["year"] < ay)]
        group = "Treated"
    else:
        pre = grp[(grp["year"] >= median_adoption_year - 5) & (grp["year"] < median_adoption_year)]
        group = "Control"
    pre = pre.sort_values("year")
    if len(pre) >= 2 and pre["gdp_per_capita"].iloc[0] > 0:
        growth = (pre["gdp_per_capita"].iloc[-1] / pre["gdp_per_capita"].iloc[0]) ** (1 / max(len(pre) - 1, 1)) - 1
        rows.append({"iso3": iso3, "group": group, "pre_period_gdp_growth": growth})

growth_df = pd.DataFrame(rows)
summary = growth_df.groupby("group")["pre_period_gdp_growth"].agg(["mean", "std", "count"])
print(summary.round(4))

from scipy.stats import ttest_ind
t_treated = growth_df.loc[growth_df.group == "Treated", "pre_period_gdp_growth"]
t_control = growth_df.loc[growth_df.group == "Control", "pre_period_gdp_growth"]
tstat, pval = ttest_ind(t_treated, t_control, equal_var=False)
print(f"\nt-test (treated vs control pre-period GDP growth): t={tstat:.3f}, p={pval:.4f}")

summary.to_csv(TABLE_DIR / "mechanism_income_check.csv")
print()
print("Saved: mechanism_income_check.csv")

A1 - Pre-period GDP per capita growth: treated vs. control


           mean     std  count
group                         
Control  0.0234  0.0379    145
Treated  0.0258  0.0251     41

t-test (treated vs control pre-period GDP growth): t=0.492, p=0.6235

Saved: mechanism_income_check.csv


In [3]:
print("=" * 60)
print("A2 - Event study with log(GDP per capita) as the outcome")
print("=" * 60)

panel_a["log_gdp_per_capita"] = np.log(panel_a["gdp_per_capita"])
income_covars = ["population", "urban_pct"]

es_income = run_event_study(
    panel_a, treatment_col="treat_carbon_price", outcome_col="log_gdp_per_capita",
    unit_col="iso3", time_col="year", covariate_cols=income_covars, window=(-5, 5),
)
print(es_income.to_string(index=False))
print()
print("Pre-trend test (income outcome):", pretrend_test(es_income))

es_income.to_csv(TABLE_DIR / "mechanism_income_event_study.csv", index=False)
print()
print("Interpretation: if income shows the same rising-then-flattening pre-trend")
print("as emissions did, that is consistent with income convergence as a shared")
print("driver of both series, rather than carbon-pricing-specific anticipation.")
print("If income shows no such pattern, the emissions pre-trend is not simply")
print("an income-convergence artifact and remains otherwise unexplained.")

A2 - Event study with log(GDP per capita) as the outcome


 event_time  estimate  std_error  ci_lower  ci_upper  p_value  n_cohorts
         -5 -0.087958   0.026052 -0.139019 -0.036896 0.000735         16
         -4 -0.013967   0.010096 -0.033754  0.005821 0.166539         16
         -3 -0.003767   0.006957 -0.017403  0.009868 0.588153         16
         -2 -0.001332   0.003290 -0.007780  0.005116 0.685579         17
         -1  0.000000   0.000000  0.000000  0.000000      NaN         18
          0  0.000264   0.003278 -0.006160  0.006688 0.935803         18
          1  0.005237   0.006056 -0.006632  0.017107 0.387143         18
          2  0.008769   0.009869 -0.010573  0.028111 0.374237         17
          3 -0.000237   0.012542 -0.024819  0.024345 0.984896         16
          4 -0.016984   0.012722 -0.041917  0.007950 0.181862         15
          5 -0.009927   0.029670 -0.068078  0.048224 0.737936         14

Pre-trend test (income outcome): {'n_pretrend_periods': 4, 'n_significant': 1, 'all_insignificant': False, 'joint_chi2': 20

---
## Part B: Does the Pattern Generalize? Carbon-Tax-Only Treatment

Re-running the complete diagnostic workflow (naive C&S + event study -> joint
pre-trend test -> Rambachan-Roth bounds -> unit-specific trends) on a
genuinely different treatment definition: carbon tax adoption only (26
countries, excludes ETS entirely), still from the same vetted WCPD data.

In [4]:
print("=" * 60)
print("B0 - Building the carbon-tax-only treatment")
print("=" * 60)

treatment_df = pd.read_csv(PROCESSED_DIR / "treatment_matrix.csv")
tax_adoption = treatment_df.loc[treatment_df["any_tax"] == 1].groupby("iso3")["year"].min()

panel_b = pd.read_csv(PROCESSED_DIR / "analysis_panel.csv")
panel_b["treat_tax_only"] = ((panel_b["any_tax"] == 1)).astype(int) if "any_tax" in panel_b.columns else np.nan
if "any_tax" not in panel_b.columns:
    panel_b = panel_b.merge(treatment_df[["iso3", "year", "any_tax"]], on=["iso3", "year"], how="left")
    panel_b["treat_tax_only"] = panel_b["any_tax"].fillna(0).astype(int)
panel_b["adoption_year_tax"] = panel_b["iso3"].map(tax_adoption)
panel_b = panel_b.dropna(subset=["log_co2"] + COVARIATES)

n_tax_treated = panel_b.loc[panel_b["treat_tax_only"] == 1, "iso3"].nunique()
print(f"N={len(panel_b):,}, countries={panel_b['iso3'].nunique()}, tax-treated={n_tax_treated}")

B0 - Building the carbon-tax-only treatment
N=6,247, countries=193, tax-treated=26


In [5]:
print("=" * 60)
print("B1 - Callaway-Sant'Anna (carbon tax only)")
print("=" * 60)

cs_tax = run_callaway_santanna(
    panel_b, outcome_col="log_co2", cohort_col="adoption_year_tax",
    unit_col="iso3", time_col="year", covariate_cols=COVARIATES,
    control_group="never_treated",
)
print(f"Overall ATT: {cs_tax.overall_att:.4f}  SE={cs_tax.overall_se:.4f}  "
      f"95% CI [{cs_tax.overall_ci_lower:.4f}, {cs_tax.overall_ci_upper:.4f}]  "
      f"p={cs_tax.overall_p_value:.4f}")
print(f"Treated units: {cs_tax.n_treated_units}  Control units: {cs_tax.n_control_units}")

B1 - Callaway-Sant'Anna (carbon tax only)


Computing ATTgt [workers=1]     0%|                    | 0/512 [00:00<?, ?it/s]

Computing ATTgt [workers=1]     4%|▊                   | 20/512 [00:00<00:02, 173.69it/s]

Computing ATTgt [workers=1]     8%|█▌                  | 39/512 [00:00<00:02, 178.43it/s]

Computing ATTgt [workers=1]    12%|██▍                 | 63/512 [00:00<00:02, 194.49it/s]

...

Computing ATTgt [workers=1]    90%|█████████████████▉  | 460/512 [00:02<00:00, 165.03it/s]

Computing ATTgt [workers=1]    93%|██████████████████▋ | 477/512 [00:02<00:00, 159.86it/s]

Computing ATTgt [workers=1]    96%|███████████████████▎| 494/512 [00:02<00:00, 160.37it/s]

Computing ATTgt [workers=1]   100%|████████████████████| 512/512 [00:02<00:00, 178.58it/s]

Overall ATT: -0.2323  SE=0.1356  95% CI [-0.4981, 0.0335]  p=0.0867
Treated units: 26  Control units: 167


In [6]:
print("=" * 60)
print("B2 - Sun-Abraham event study (carbon tax only)")
print("=" * 60)

es_tax = run_event_study(
    panel_b, treatment_col="treat_tax_only", outcome_col="log_co2",
    unit_col="iso3", time_col="year", covariate_cols=COVARIATES, window=(-5, 5),
)
print(es_tax.to_string(index=False))
print()
pt_tax = pretrend_test(es_tax)
print("Pre-trend test (tax only):", pt_tax)

es_tax.to_csv(TABLE_DIR / "tax_only_event_study.csv", index=False)

B2 - Sun-Abraham event study (carbon tax only)


 event_time  estimate  std_error  ci_lower  ci_upper      p_value  n_cohorts
         -5  0.292628   0.047704  0.199130  0.386126 8.556707e-10         14
         -4  0.080375   0.015312  0.050364  0.110387 1.529225e-07         14
         -3  0.066535   0.010316  0.046316  0.086755 1.122064e-10         14
         -2  0.023403   0.005615  0.012397  0.034408 3.075311e-05         15
         -1  0.000000   0.000000  0.000000  0.000000          NaN         16
          0 -0.020404   0.005887 -0.031942 -0.008865 5.288099e-04         16
          1 -0.053689   0.006632 -0.066687 -0.040691 5.697407e-16         16
          2 -0.044456   0.009991 -0.064038 -0.024873 8.611851e-06         15
          3 -0.075203   0.013548 -0.101757 -0.048649 2.845886e-08         15
          4 -0.101617   0.016613 -0.134178 -0.069057 9.544674e-10         14
          5 -0.321608   0.059988 -0.439182 -0.204035 8.266118e-08         13

Pre-trend test (tax only): {'n_pretrend_periods': 4, 'n_significant': 4, 'a

In [7]:
print("=" * 60)
print("B3 - Rambachan-Roth bounds (carbon tax only)")
print("=" * 60)

info = es_tax.attrs["event_time_cov_matrix"]
betahat = es_tax.set_index("event_time").loc[info["event_times"], "estimate"].values
sigma = info["sigma"]
numPre = sum(1 for t in info["event_times"] if t < 0)
numPost = sum(1 for t in info["event_times"] if t >= 0)

l_vec = honestdid.basis_vector(index=numPost, size=numPost)
bounds_tax = honestdid.createSensitivityResults_relativeMagnitudes(
    betahat=betahat, sigma=sigma, numPrePeriods=numPre, numPostPeriods=numPost,
    Mbarvec=[0.5, 1.0, 1.5, 2.0], l_vec=l_vec,
)
print(f"Target: k={numPost - 1} (largest post-period)")
print(bounds_tax.to_string(index=False))
bounds_tax.to_csv(TABLE_DIR / "tax_only_rambachan_roth_bounds.csv", index=False)

B3 - Rambachan-Roth bounds (carbon tax only)


Target: k=5 (largest post-period)
       lb       ub method   Delta  Mbar
-1.199758 0.481585   C-LF DeltaRM   0.5
-1.199758 1.199758   C-LF DeltaRM   1.0
-1.199758 1.199758   C-LF DeltaRM   1.5
-1.199758 1.199758   C-LF DeltaRM   2.0


In [8]:
print("=" * 60)
print("B4 - Unit-specific linear trends (carbon tax only)")
print("=" * 60)

es_tax_trends = run_event_study(
    panel_b, treatment_col="treat_tax_only", outcome_col="log_co2",
    unit_col="iso3", time_col="year", covariate_cols=COVARIATES, window=(-5, 5),
    unit_specific_trends=True,
)
print(es_tax_trends.to_string(index=False))
print()
print("Pre-trend test (tax only, unit trends):", pretrend_test(es_tax_trends))
es_tax_trends.to_csv(TABLE_DIR / "tax_only_unit_trends.csv", index=False)

B4 - Unit-specific linear trends (carbon tax only)


 event_time  estimate  std_error  ci_lower  ci_upper  p_value  n_cohorts
         -5  0.021878   0.036578 -0.049813  0.093569 0.549757         14
         -4  0.001810   0.014703 -0.027008  0.030627 0.902042         14
         -3  0.008605   0.011046 -0.013044  0.030255 0.435946         14
         -2 -0.010653   0.007921 -0.026178  0.004872 0.178647         15
         -1  0.000000   0.000000  0.000000  0.000000      NaN         16
          0 -0.005166   0.005168 -0.015296  0.004964 0.317514         16
          1 -0.008116   0.007906 -0.023612  0.007380 0.304634         16
          2  0.016344   0.009521 -0.002317  0.035005 0.086052         15
          3  0.002045   0.014178 -0.025742  0.029833 0.885303         15
          4 -0.002529   0.017385 -0.036604  0.031545 0.884320         14
          5  0.093367   0.035640  0.023515  0.163220 0.008800         13

Pre-trend test (tax only, unit trends): {'n_pretrend_periods': 4, 'n_significant': 0, 'all_insignificant': True, 'joint_chi

---
## Summary

In [9]:
print("=" * 60)
print("SUMMARY: Does the pattern generalize to carbon-tax-only treatment?")
print("=" * 60)
print(f"Naive C&S ATT (tax only): {cs_tax.overall_att:.4f}, p={cs_tax.overall_p_value:.4f}")
print(f"Naive joint pre-trend test (tax only): p={pt_tax['joint_p_value']:.4g}, "
      f"passes={pt_tax['joint_test_passes']}")
print(f"Rambachan-Roth bound at Mbar=0.5 (tax only, largest post-period): "
      f"[{bounds_tax.iloc[0]['lb']:.4f}, {bounds_tax.iloc[0]['ub']:.4f}]")
print()
print("Proceed to manuscript update with these generalization and mechanism results")

SUMMARY: Does the pattern generalize to carbon-tax-only treatment?
Naive C&S ATT (tax only): -0.2323, p=0.0867
Naive joint pre-trend test (tax only): p=2.042e-12, passes=False
Rambachan-Roth bound at Mbar=0.5 (tax only, largest post-period): [-1.1998, 0.4816]

Proceed to manuscript update with these generalization and mechanism results
